![image_1787419385294.png](./image_1787419385294.png "image_1787419385294.png")

# Contexto
En el sector asegurador, la detección temprana de posibles casos de fraude permite focalizar esfuerzos de investigación, proteger la sostenibilidad técnica del portafolio y mejorar la eficiencia operativa de los equipos de siniestros, auditoría y gestión del riesgo. Esta prueba busca evaluar la capacidad del candidato para transformar una necesidad de negocio en un problema analítico abordable, construir un modelo predictivo con datos reales o simulados, interpretar sus resultados y comunicar sus hallazgos de forma clara para audiencias técnicas y de negocio.


#Reto técnico
Construir una solución analítica para estimar la probabilidad de que un registro, siniestro, reclamación, transacción o caso del negocio asegurador corresponda a un posible fraude. El candidato deberá trabajar con una única matriz de datos suministrada por la compañía, cuya variable objetivo es FraudeS /N, y desarrollar un flujo completo de análisis y modelado, desde la exploración inicial hasta la sustentación de resultados.
Se dará puntos adicionales si la solución es desarrollada en Databricks Free Edition, en caso contrario deberá ser implementada en Python, Incluyendo Git para control de versiones con un repositorio organizado.

#Configuración del repositorio
Es importante tener un versionamiento del proyecto por lo que se vincula este notebook a un repositorio previamente creado y sincronizado con databricks

#Instalación de paquetes

In [0]:
#instalamos la librerías
#para lectura de datos
%pip install -q openpyxl
#para mixed nulls
%pip install -q deepchecks --upgrade
#para estadistica descriptiva
%pip install -q "pathspec<0.12"
%pip install -q scikit-build-core cmake ninja pybind11
%pip install -q --no-build-isolation "phik==0.12.5"
%pip install -q ydata_profiling
#para catboost
%pip install -q catboost

#Reinicio del entorno

In [0]:
%restart_python

#Importación de librerías

In [0]:
#Manejo de datos
import pandas as pd
import numpy as np

if not hasattr(np, "Inf"):
    np.Inf = np.inf

from sklearn.preprocessing import LabelEncoder
#librerías gráficas
import seaborn as sns
import matplotlib.pyplot as plt

#Reconocimiento de nulos
from deepchecks.tabular.checks import MixedNulls
#Validación cruzada
from sklearn.model_selection import KFold
#Modelación
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from joblib import dump
#métricas de evaluación
from sklearn.metrics import accuracy_score
from sklearn import metrics
#hiperparametrización
from sklearn.model_selection import GridSearchCV
#Correlacion
from scipy.stats import chi2_contingency
#Descripcion estadistica
from ydata_profiling import ProfileReport

#Lectura de los datos
Hacemos la lectura de los datos que ya previamente fueron subidos al catalog de databricks, adicionalmente miramos unos cuantos registros para entender mejor la estructura de los datos y el formato de las variables

In [0]:
Ruta_base = "/Volumes/workspace/prueba_tecnica/muestra_base_fraude/Muestra_Base_fraude.xlsx"
Base = pd.read_excel(Ruta_base) 
Base.head(5)

*   Contamos con una base de **12.776 filas** y **40 columnas** , esto puede cambiar a medida de que hagamos transformaciones y limpieza.

*   Podemos apreciar el **tipo de dato** y **cantidad de no nulos** en cada variable, esto último se debe analizar junto con **conocimiento de negocio** ya que en el caso de algunas variables puede ser **normal** el tener muchos nulos
*   Adicionalmente nos podemos dar una idea del **nivel de completitud** de cada variable, (esto no quere decir que sea el escenario final de completitud , ya que los varores **nulos** pueden estar en **diferentes formatos**, no todos necesariamente detectados por la función)

In [0]:
Base.info()

#Detección de registros duplicados
Primero inspeccionamos la base virgen en busca de registros repetidos y al parecer tenemos 329 reclamos Duplicados. (Puede que mas adelante cuando tenga mas entendimiento sobre la base pueda hacer combinaciones buscando otro tipo de duplicidades) 

In [0]:
##Detección de registros duplicados
Base.duplicated().value_counts()

Hacemos una primera limpieza de **registros duplicados** los cuales pueden ensuciar nuestros futuros modelos , nos quedamos con **12447 registros unicos** , con eso nos aseguramos que cada registro es una reclamacion unica.

In [0]:
Base_sin_duplicados = Base.drop_duplicates().reset_index(drop=True)

#Entendimiento de los datos
* Segun se puede ver a simple vista esta base de datos representa un conjunto de reclamaciones de seguros de vida (rentas, invalidez, incapacidades) hechas por clientes en diferentes ventanas de tiempo , cada registro trae todo el ciclo del siniestro "desde la vigencia de la póliza hasta el cierre" junto con la etiqueta de si fue detectado como fraude o no

* Con el fin de conocer mas a fondo la naturaleza de la base y asi poder descartar todas las variables que por definición son irrelevantes para la construccion del modelo , se crea un glosario con el entendimiento de cada variable

### Producto y canal de venta

- **Ramo**: código del ramo del seguro.
- **Ramo_Desc**: descripción del ramo (debe ser homóloga a *Ramo*).
- **Nombre_plan**: producto específico al que pertenece la reclamación.
- **Codigo_Canal_Comercial_Op** / **Nombre_Canal_Comercial**: código y nombre del canal por el que se vendió el seguro.
- **Amparo_Desc**: nombre del amparo (cobertura) afectado por la reclamación.

### Póliza y asegurado

- **Fecha_Primera_Vigencia_Cert**: fecha de primera vigencia del certificado individual (me interesa mas esta fecha por que es mas exacta que la master).
- **fecha_primera_vigencia_pol**: fecha de primera vigencia de la póliza máster. En seguros individuales debería coincidir con la del certificado.
- **IDENTIFICACION_asegurado**: número de identificación del asegurado.
- **SEXO_asegurado**: género del asegurado.
- **edad_ingreso_asegurado** / **edad_actual_asegurado**: edad al vincularse a la compañía vs. edad al momento de la reclamación (gran potencial para definir antiguedad).
- **vigencia_poliza**: número de vigencias/renovaciones de la póliza; funciona como proxy de antigüedad del cliente.
- **vigencia_certificado**: similar a *vigencia_poliza* pero a nivel de certificado.
- **FEXPEDICION**: todo indica que es la fecha de expedición de la póliza/certificado, no del reclamo. Coincide de cerca con las fechas de primera vigencia, y en 91% de los casos es anterior a *FSINIESTRO* — la póliza se expide antes de que ocurra el siniestro, como debería ser.

### Estructura comercial

- **CODSUC** / **SUCURSAL**: código y nombre de la oficina.
- **REGIONAL**: regional a la que pertenece la oficina.
- **AGENTE** / **CODAG**: nombre e identificador del agente o asociación que vendió el seguro.

### El siniestro en sí

- **CAUSASTRO** / **DESCAUSA**: id y nombre de la causa de la reclamación.
- **DIAGNOSTICO**: diagnóstico médico asociado.
- **FSINIESTRO**: fecha en que ocurrió el siniestro.
- **F_Notificacion**: fecha en que se notificó el siniestro a la compañía.
- **Fecha_Recepcion**: fecha de recepción formal de la reclamación.
- **Fecha_Apertura**: fecha de apertura del caso.
- **Fecha_Primer_Cierre_Siniestro**: fecha del primer cierre. 
- **Ind_Tipo_Atencion**: canal/modalidad de atención (interna vs. externa).
- **Ind_Pago_Automatico**: si el pago se hizo de forma automática (S/N).

### Montos y estado

- **Sum(Valor_Reservas_Inicial)**: reserva inicial constituida.
- **Sum(Valor_Reservas)**: reserva final/actual.
- **Sum(Valor_Pagos)**: valor efectivamente pagado.
- **estado**: estado actual de la reclamación.
- **Cobertura**: cobertura afectada — se cruza directamente con *Amparo_Desc*.
- **Tipo apertura**: medio por el que se atendió la reclamación, muy relacionada con *Ind_Tipo_Atencion*.

### Variable objetivo y reporte

- **Fraude S /N**: variable objetivo — si la reclamación fue determinada como fraude.
- **Periodo Reporte**: mes del reporte.
- **Fecha de reporte**: fecha completa del reporte del fraude.
- **Año**: año del reporte.

# Eliminacion de variables irrelevantes por definición

- **Ramo**: como id no tiene ningun valor descriptivo, ademas ya existe su version en texto llamada Ramo_Desc.
- **Codigo_Canal_Comercial_Op**: igual que Ramo, no aporta nada como codigo y tiene su homologo descriptivo en Nombre_Canal_Comercial.
- **CODSUC**: el id en si no tiene valor descriptivo, ya existe su homologo SUCURSAL con el nombre de la oficina.
- **CODAG**: comparte la misma descripcion que AGENTE, asi que se queda solo este ultimo.
- **CAUSASTRO**: es un id sin valor descriptivo, su homologo con descripcion es DESCAUSA.

 NOTA: a pesar que la variable IDENTIFICACION_asegurado tiene la misma naturaleza que estas variables , aun no la elimino por que a partir de ella puedo crear otras variables , ademas esta variable es fijo  (data leakage) debido a la identidad y cardinalidad

Eliminamos las variables anteriormente mencionadas, quedamos con 35 de las 40 variables originales

In [0]:
columnas_a_eliminar = ['Ramo', 'Codigo_Canal_Comercial_Op', 'CODSUC', 'CODAG', 'CAUSASTRO']

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_a_eliminar)

print(f"Columnas eliminadas: {columnas_a_eliminar}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

# Eliminacion de variables redundantes por definición

- **fecha_primera_vigencia_pol**: comparte gran parte de sus valores con Fecha_Primera_Vigencia_Cert (un 68% de sus valores). Me quedo con esta ultima porque da mayor detalle del cliente al ser la vigencia del certificado y no de la poliza master.
- **vigencia_poliza**: comparte varios valores con vigencia_certificado (en un 76%), y al igual que en el caso anterior prefiero quedarme con la version de certificado.
- **F_Notificacion**, **Fecha_Recepcion**, **Fecha_Apertura**: estas tres fechas son casi identicas entre si. Me quedo con **F_Notificacion** porque el nombre es mas diciente y hace referencia directa a la fecha de aviso del siniestro.
- **Periodo Reporte**, **Fecha de reporte**, **Año**: estas tres hacen referencia a la misma fecha en distintos niveles de detalle, asi que me quedo unicamente con **Fecha de reporte** por ser la mas completa.

A continuacion mostramos las coincidencias entre variables mencionadas en el texto anteior, que evidencian su redundancia

In [0]:
coincidencia_vigencias = (Base['Fecha_Primera_Vigencia_Cert'] == Base['fecha_primera_vigencia_pol']).mean()

print(f"Porcentaje de coincidencia exacta entre ambas fechas: {coincidencia_vigencias:.2%}")

In [0]:
coincidencia_vigencia_num = (Base['vigencia_poliza'] == Base['vigencia_certificado']).mean()

print(f"Porcentaje de coincidencia exacta entre ambas vigencias: {coincidencia_vigencia_num:.2%}")

In [0]:
coincidencia_notif_recep = (Base['F_Notificacion'] == Base['Fecha_Recepcion']).mean()
coincidencia_notif_apert = (Base['F_Notificacion'] == Base['Fecha_Apertura']).mean()
coincidencia_recep_apert = (Base['Fecha_Recepcion'] == Base['Fecha_Apertura']).mean()
coincidencia_las_tres = ((Base['F_Notificacion'] == Base['Fecha_Recepcion']) & 
                          (Base['Fecha_Recepcion'] == Base['Fecha_Apertura'])).mean()

print(f"F_Notificacion == Fecha_Recepcion: {coincidencia_notif_recep:.2%}")
print(f"F_Notificacion == Fecha_Apertura: {coincidencia_notif_apert:.2%}")
print(f"Fecha_Recepcion == Fecha_Apertura: {coincidencia_recep_apert:.2%}")
print(f"Las tres coinciden al mismo tiempo: {coincidencia_las_tres:.2%}")

In [0]:
meses = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',6:'Junio',7:'Julio',
         8:'Agosto',9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}

Base['mes_de_fecha_reporte'] = Base['Fecha de reporte'].dt.month.map(meses)

coincidencia_mes = (Base['mes_de_fecha_reporte'] == Base['Periodo Reporte']).mean()
coincidencia_anio = (Base['Fecha de reporte'].dt.year == Base['Año']).mean()

print(f"Mes de Fecha de reporte == Periodo Reporte: {coincidencia_mes:.2%}")
print(f"Año de Fecha de reporte == Año: {coincidencia_anio:.2%}")

Por último eliminamos las variables consideradas redundantes por definición, quedando con 29 de las 40 variables originales

In [0]:
columnas_redundantes = [
    'fecha_primera_vigencia_pol',   # se conserva Fecha_Primera_Vigencia_Cert
    'vigencia_poliza',              # se conserva vigencia_certificado
    'Fecha_Recepcion',              # se conserva F_Notificacion
    'Fecha_Apertura',               # se conserva F_Notificacion
    'Periodo Reporte',              # se conserva Fecha de reporte
    'Año'                           # se conserva Fecha de reporte
]

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_redundantes)

print(f"Columnas eliminadas: {columnas_redundantes}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

#Revisión variables categóricas
Revisamos las categorias de estas variables con el fin de:
- Encontrar categorías que significan lo mismo pero que estan escritas diferente
- Encontrar categorías que se puedan interpretar como NAN


In [0]:
columnas = [
    'Ramo_Desc', 'Nombre_plan',
    'Nombre_Canal_Comercial','SEXO_asegurado',
    'SUCURSAL', 'REGIONAL', 'AGENTE',
    'DESCAUSA', 'DIAGNOSTICO', 'Ind_Tipo_Atencion', 'Ind_Pago_Automatico',
    'estado', 'Cobertura', 'Tipo apertura', 'Fraude S /N'
]

for col in columnas:
    valores = Base_sin_duplicados[col].unique()
    print(f"\n{'='*60}")
    print(f"Columna: {col}  |  Valores únicos: {len(valores)}")
    print(f"{'='*60}")
    if len(valores) <= 100:
        print(valores)
    else:
        print(f"(demasiados para mostrar todos, primeros 20): {valores[:100]}")


Luego de examinar las variables con menor cardinalidad pude encontrar que en la variable **estado** cuneta con las opciones  **Anulado / ANULADO** , **Objetado / OBJETADO** y **Tramitado / TRAMITADO** las cuales con categorias que significan lo mismo pero estan escritas de manera diferente , lo mismo para la variable **Tipo apertura** que cuenta con las opciones **Oficina / oficina** , Con el fin de unificar dichas categorias , Paso a convertir todas las categorias de estas variables en mayúsculas 

In [0]:
Base_sin_duplicados['estado'] = Base_sin_duplicados['estado'].str.strip().str.upper()
Base_sin_duplicados['Tipo apertura'] = Base_sin_duplicados['Tipo apertura'].str.strip().str.upper()

print(Base_sin_duplicados['estado'].value_counts())
print(Base_sin_duplicados['Tipo apertura'].value_counts())


#Unificamos los valores nulos

Adicionalmente identificamos **diferentes valores** que pueden ser interpredados como nulos , esto ayuda bastante para poder medir com mayor exactitud que tan completas estan las variables.

In [0]:
Base_sin_duplicados.replace(["none","None", "null","NaN","nan","[]","{}",""," ","not_specified",None,"Sin Información","?"], np.nan, inplace=True)
pd.set_option('display.max_rows', None)

Porcetajes_na = Base_sin_duplicados.isnull().mean() * 100

df_porcentaje_nan = pd.DataFrame({'variable':Porcetajes_na.index, 'Porcentaje_na':Porcetajes_na.values})

df_porcentaje_nan.head(115)


Luego de unificar los **valores nulos** el panorama inicial no cambia mucho , ninguna variable supera el 1% de **datos nulos** detectados por el momento , por lo que afortunadamente hasta este momento todas las variables cuentan con la informacion suficiente , sin embargo  es necesario limpiar los pocos registros que contienen estos **datos nulos** , en total son solo **218 registros** por eliminar por el momento asi que procedo y quedamos con **12.229 registros** 

In [0]:
Base_sin_duplicados.isnull().any(axis=1).sum()

Base_sin_duplicados = Base_sin_duplicados.dropna()

print(f"Registros despues de eliminar: {len(Base_sin_duplicados)}")

#Revisión data leakage
Dado que el objetivo del modelo es categorizar posibles perfiles de fraude primero debemos analizar que variables estarían desponibles antes de que en la base se concluyera como caso de fraude o no , por lo que a continuacion procedo con el analisis y toma de accion pertinente según el caso

### Estado

Por definición, esta variable determina si un reclamo fue admitido o no. Como estamos trabajando con una base de datos en la que ya se determinó qué reclamaciones fueron fraude y cuáles no, lo más lógico es que esas reclamaciones hayan quedado como Objetadas o Anuladas —algo que no se podría saber sin antes haber determinado que se trataba de fraude—, y eso es justamente lo que muestran los datos: estas categorías están relacionadas en un 80% con casos de fraude. Por lo tanto, lo mejor es eliminar esta variable para evitar el riesgo de fuga de información.

In [0]:
es_objetado_anulado = Base_sin_duplicados['estado'].isin(['OBJETADO', 'ANULADO'])

tasa_fraude_objetado_anulado = Base_sin_duplicados[es_objetado_anulado]['Fraude S /N'].eq('Fraude').mean()
tasa_no_fraude_resto = Base_sin_duplicados[~es_objetado_anulado]['Fraude S /N'].eq('No es fraude').mean()

print(f"Cuando estado es Objetado o Anulado, tasa de fraude: {tasa_fraude_objetado_anulado:.1%}")
print(f"Cuando estado NO es Objetado ni Anulado, tasa de no-fraude: {tasa_no_fraude_resto:.1%}")

### Fecha_Primer_Cierre_Siniestro

Esta variable solo toma un valor real cuando el caso ya se cerro. Trabajando sobre una base donde ya se sabe que reclamaciones fueron fraude y cuales no, lo logico es que los casos ya resueltos sean los que mas se relacionan con fraude, algo que no se podria saber sin antes conocer el resultado del caso. Los datos lo confirman: los casos con fecha de cierre real tienen una tasa de fraude del 67.9%, contra 38.5% en los que aun estan abiertos, una diferencia de casi 30 puntos. Por eso se elimina, mismo riesgo de fuga de informacion que las anteriores.

In [0]:
fecha_cierre_parseada = pd.to_datetime(Base_sin_duplicados['Fecha_Primer_Cierre_Siniestro'], dayfirst=True, errors='coerce')
caso_cerrado = fecha_cierre_parseada.notnull() & (fecha_cierre_parseada.dt.year != 1900)

tasa_fraude_cerrado = Base_sin_duplicados[caso_cerrado]['Fraude S /N'].eq('Fraude').mean()
tasa_fraude_no_cerrado = Base_sin_duplicados[~caso_cerrado]['Fraude S /N'].eq('Fraude').mean()

print(f"Tasa de fraude cuando el caso tiene fecha de cierre real: {tasa_fraude_cerrado:.1%}")
print(f"Tasa de fraude cuando el caso NO tiene fecha de cierre real: {tasa_fraude_no_cerrado:.1%}")

### Sum(Valor_Pagos) y Sum(Valor_Reservas)

Estas variables tambien estan muy ligadas al resultado final del veredicto, dado que un siniestro calificado como fraude generalmente no se paga (hay varios valores de pago en 0). El valor final de la reserva tambien se ve afectado por esto: cuando el caso se cierra sin pago (por rechazo o anulacion), la reserva se libera igual, asi como cuando si hay pago tambien se libera. Es decir, la reserva liberada no distingue directamente si el caso fue pagado o rechazado, sino simplemente si el caso ya se cerro.

Mirando la relacion entre el no pago y los casos de fraude, encontramos una tasa del 65.6%, no tan alta como la de la variable estado, pero al separar la reserva liberada segun si hubo pago o no, el panorama se aclara mas: cuando se libera con pago, la tasa de fraude es de apenas 57.6% (casi neutral), pero cuando se libera sin pago (rechazo o anulacion) la tasa sube a 81.1%. Esto confirma que la reserva no mide fraude por si misma, sino si el caso ya fue resuelto, y los casos de fraude en esta base se resuelven con mas frecuencia (83.6%) que los legitimos (62.5%). Por esta razon, junto con el hecho de que ambas variables solo toman su valor definitivo despues de que el caso ya fue investigado, se consideran variables con riesgo de fuga de informacion (data leakage) y bajo este criterio seran eliminadas

In [0]:
pago_cero = Base_sin_duplicados['Sum(Valor_Pagos)'] == 0

tasa_fraude_pago_cero = Base_sin_duplicados[pago_cero]['Fraude S /N'].eq('Fraude').mean()
tasa_fraude_con_pago = Base_sin_duplicados[~pago_cero]['Fraude S /N'].eq('Fraude').mean()

print(f"Tasa de fraude cuando el pago es 0: {tasa_fraude_pago_cero:.1%}")
print(f"Tasa de fraude cuando SI hubo pago: {tasa_fraude_con_pago:.1%}")

In [0]:
reserva_cero = Base_sin_duplicados['Sum(Valor_Reservas)'] == 0
pago_cero = Base_sin_duplicados['Sum(Valor_Pagos)'] == 0

print("Tasa de fraude - reserva liberada con pago:", Base_sin_duplicados[reserva_cero & ~pago_cero]['Fraude S /N'].eq('Fraude').mean())
print("Tasa de fraude - reserva liberada sin pago:", Base_sin_duplicados[reserva_cero & pago_cero]['Fraude S /N'].eq('Fraude').mean())
print("Tasa de fraude - reserva no liberada:", Base_sin_duplicados[~reserva_cero]['Fraude S /N'].eq('Fraude').mean())

### Ind_Pago_Automatico

Al igual que las demás variables, considero que esta está muy ligada al evento del pago; es decir, sin que se haya realizado el pago no es posible determinar si este fue manual o automático. Por eso considero que esta variable se construye después de la clasificación de fraude. Otra evidencia de esto es su relación con la variable **estado** (que ya identificamos como fuerte data leakage): existe una relación de casi el 100% entre los pagos no automáticos y los siniestros en estado Anulado u Objetado. Por lo tanto, esta variable también corre el riesgo de ser data leakage y debe ser eliminada

In [0]:
categorias_resolucion = ['ANULADO', 'OBJETADO']
casos_resolucion = Base_sin_duplicados[Base_sin_duplicados['estado'].isin(categorias_resolucion)]

print(casos_resolucion['Ind_Pago_Automatico'].value_counts(normalize=True))

Procedemos con la eliminacion de las variables con riesgo data leakage y nos quedamos con 25 de las 40 columnas con que iniciamos el análisis

In [0]:
columnas_leakage = ['estado', 'Sum(Valor_Pagos)', 'Sum(Valor_Reservas)', 'Ind_Pago_Automatico','Fecha_Primer_Cierre_Siniestro']

print(f"Registros antes: {Base_sin_duplicados.shape}")

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_leakage)

print(f"Columnas eliminadas: {columnas_leakage}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

#Estadística Descriptiva
Con la función **ProfileReport** podemos implementar un **análisis descriptivo** profundo tanto para las variables numéricas como categóricas , aprovecharemos esta visual para conocer la **distrubución** de nuestra variables, **calidad** y **atípicos** , esto nos va a permitir refinar aun mas la limpieza de registros y variables que empezamos en las secciones anteriores

In [0]:
descripcion_estadistica = ProfileReport(Base_sin_duplicados, title="Profiling Report")
descripcion_estadistica

### Hallazgos de la Estadística Descriptiva

1. Tras haber eliminado gran cantidad de columnas, nos encontramos con 243 registros duplicados, los cuales serán eliminados.

2. Se elimina la variable **Amparo_Desc**, ya que guarda una correlación alta con las variables **Cobertura** y **Ramo**. Con **Cobertura** basta, y además presenta menor cardinalidad.

3. Las variables **edad_actual_asegurado** y **edad_ingreso_asegurado** guardan una similitud del 93%. Por esta razón me quedo únicamente con **edad_actual_asegurado**. Pensaba usar ambas variables para calcular la antigüedad, pero dado que son tan similares y ya cuento con **vigencia_certificado**, opto por eliminar una de las dos.

4. Existe una correlación alta entre **Ind_Tipo_Atencion** y **Tipo apertura**; al parecer explican lo mismo, pero de forma diferente. Me quedo con **Tipo apertura** porque resulta mucho más explicativa.



Las variables **DIAGNOSTICO** , **AGENTE** , **SUCURSAL** y **Nombre_Canal_Comercial** tienen una alta cardinalidad por lo que la libreria las excluyo del analisis de correlacion , por ello decidi hacer aparte una matriz de correlacion para examinalas frente a las variables que por definición guardaria una correlación alta, vamos a validar esa hipotesis

In [0]:
import phik

columnas_correlacion = ['DESCAUSA', 'DIAGNOSTICO', 'AGENTE', 'SUCURSAL', 
                         'REGIONAL', 'Nombre_Canal_Comercial', 'Fraude S /N']

matriz_alta_cardinalidad = Base_sin_duplicados[columnas_correlacion].phik_matrix(interval_cols=[])

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_alta_cardinalidad, annot=True, fmt='.3f', cmap='coolwarm', 
            vmin=0, vmax=1, square=True, linewidths=0.5)
plt.title("Correlacion (phik) - Variables de alta cardinalidad vs Fraude S /N")
plt.tight_layout()
plt.show()

### Hallazgos matriz de correlación

**DESCAUSA vs DIAGNOSTICO:** Ambas variables tienen una correlacion de 0.9997 entre si, practicamente identicas. Sin embargo, DIAGNOSTICO tiene una relacion mucho mas fuerte con la variable objetivo (0.837) que DESCAUSA (0.572). Aunque DIAGNOSTICO tiene mucha mas cardinalidad (1478 categorias contra 28 de DESCAUSA), se conserva DIAGNOSTICO y se elimina DESCAUSA, porque se pierde muy poco en simplicidad y se gana bastante poder predictivo.



**AGENTE:** Se elimina por dos razones. Primero, tiene una cardinalidad muy alta (1079 categorias), lo que implica mas riesgo de sobreajuste y mayor costo computacional en los proximos modelos y tiene una alta correlacion con SUCURSAL, canal comercial y regional asi que no aporta informacion nueva que estas otras no cubran.


**SUCURSAL, REGIONAL y Nombre_Canal_Comercial:** Las tres variables estan altamente correlacionadas entre si (SUCURSAL-REGIONAL: 1.000, SUCURSAL-Nombre_Canal_Comercial: 0.999). Comparando cada una contra la variable objetivo:

- SUCURSAL: 0.672
- Nombre_Canal_Comercial: 0.476
- REGIONAL: 0.284

Se conserva unicamente SUCURSAL, ya que es la que mayor relacion tiene con el fraude. Eliminar REGIONAL y Nombre_Canal_Comercial no representa perdida real de informacion, porque esta ya esta contenida en SUCURSAL, que ademas ofrece un nivel de detalle mas especifico.

Tras los hallazgos encontrados, procedemos a eliminar las variables ya mencionadas., tas esto nos quedamos con 18 variables de las 40 originales

In [0]:
columnas_correlacionadas = [
    'Amparo_Desc',
    'edad_ingreso_asegurado',
    'Ind_Tipo_Atencion',
    'DESCAUSA',
    'AGENTE',
    'REGIONAL',
    'Nombre_Canal_Comercial'
]

print(f"Dimensiones antes: {Base_sin_duplicados.shape}")

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_correlacionadas)

print(f"Columnas eliminadas: {columnas_correlacionadas}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

Adicionalmente tras haber modificado la base eliminamos registros duplicados , nos quedamos con 11982 registros

In [0]:
print(f"Registros antes de eliminar duplicados: {len(Base_sin_duplicados)}")
print(f"Duplicados encontrados: {Base_sin_duplicados.duplicated().sum()}")

Base_sin_duplicados = Base_sin_duplicados.drop_duplicates()

print(f"Registros despues de eliminar duplicados: {len(Base_sin_duplicados)}")

**Fecha_Primer_Cierre_Siniestro**: segun la libreria del analisis estadistico esta variable tiene problemas que no fueron detectados en ninguno de los analisis anteriores a continuacion se explora esta misma

#Creacion de nuevas variables 
Usando como insumo variables que no generan valor por si solas creamos nuevas variables que tendrán un gran potencial en la modelación

##Unificación de valores nulos
**Fecha_Primer_Cierre_Siniestro**: fecha del primer cierre. Ojo: hay registros con *1900-01-01*, que es un placeholder de "sin cerrar" y no una fecha real — hay que tratarlo como nulo.

#Creacion de nuevas variables

#Correr el modelo